# Notebook 01 — `gp_only` mode: Fisher forecast on the GP emulator

## Prerequisites

1. **PYTHONPATH** must include `src/` of this repo and the upstream
   `lya_emulator_full` clone:
   ```
   export PYTHONPATH=/path/to/lya_emulator_full:<repo_root>/src
   ```
   On Greatlakes: `lya_emulator_full` is at
   `/home/mfho/student_projects/lya_emulator_full`.
2. **`data/kodiaq_gp/`** must exist (it is committed to the repo).
   No other data is needed for `gp_only`.

## What this notebook does

Runs the `gp_only` pipeline at z = 3.6 using the KODIAQ-SQUAD covariance
and reads the resulting `σ_GP` per parameter.

In `gp_only` mode the forward model is the GP emulator itself — no PySR,
no symbolic equations. This gives the **reference constraint σ_GP**:
the best the data can do given a perfect (but opaque) emulator.

### The Fisher formula recap

```
F_ij = (∂P_F/∂θ_i)^T  C^{-1}  (∂P_F/∂θ_j)
σ_i  = sqrt( (F^{-1})_{ii} )
```

Derivatives are 5-point centered stencils with adaptive step halving
(see `src/priya_forecast/fisher.py`). `C` is the KODIAQ-SQUAD measured
covariance (Karaçaylı et al. 2021).

In [ ]:
import sys
from pathlib import Path

# Adjust these paths to match your environment.
REPO_ROOT = Path("../").resolve()          # parent of this notebooks/ dir
LYA_EMU_ROOT = Path("/home/mfho/student_projects/lya_emulator_full")

for p in [str(REPO_ROOT / "src"), str(LYA_EMU_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path configured.")

## 1. Build a `PipelineConfig` for `gp_only`

The easiest way is to load the example YAML and override the mode.
Alternatively, construct `PipelineConfig` directly.

In [ ]:
from priya_forecast.single_z.config import load_config, PipelineConfig

# Option A: load the example YAML (mode is already gp_only there)
cfg = load_config(REPO_ROOT / "configs/single_z/example.yaml")
print(f"mode     : {cfg.mode}")
print(f"redshift : {cfg.redshift}")
print(f"data src : {cfg.data.source}")
print(f"output   : {cfg.output_dir}")
print(f"params   : {cfg.parameters}")

In [ ]:
# Option B: build programmatically (useful for tweaking individual fields)
from priya_forecast.single_z.config import (
    PipelineConfig, DataConfig, GPConfig, KRange,
)

cfg_b = PipelineConfig(
    mode="gp_only",
    redshift=3.6,
    output_dir="/tmp/nb01_gp_only/",
    data=DataConfig(source="kodiaq", conservative=True, mock_data="gp"),
    gp=GPConfig(basedir=str(REPO_ROOT / "data/kodiaq_gp")),
    k_range=KRange(min=0.001, max=0.04),
)
cfg_b.validate()   # raises ValueError on any misconfiguration
print("cfg_b validates OK")

## 2. Run the pipeline

`priya_forecast.single_z.pipeline.run(cfg)` dispatches on `cfg.mode`.
For `gp_only` it:
1. Builds the GP emulator from `cfg.gp.basedir`.
2. Builds a `KSDataLikelihood` wrapping the GP.
3. Calls `fisher_matrix` with a 5-point stencil per parameter.
4. Writes `forecast_table.txt` and `scorecard.md` into `cfg.output_dir`.
5. Returns a result dict with `sigma_gp`, `fisher`, `table_path`, etc.

This takes roughly **1–3 minutes** on a single core (5 GP evaluations per
parameter × 11 parameters × stencil halving iterations).

In [ ]:
from priya_forecast.single_z.pipeline import run

# Use cfg_b so outputs land in /tmp.
result = run(cfg_b)

print("\nResult keys:", list(result.keys()))

## 3. Inspect σ_GP

`result["sigma_gp"]` is a numpy array of shape `(n_params,)` — one
marginalized 1σ error per parameter in the canonical PRIYA parameter order.

In [ ]:
import numpy as np
from priya_forecast.parameters import PARAMS_11D

sigma_gp = result["sigma_gp"]
selected = result["selected_params"]   # tuple of Param objects

print(f"{'param':<12s}  {'fid':>8s}  {'σ_GP':>10s}  {'σ_GP/|fid|':>12s}")
print("-" * 48)
for p, s in zip(selected, sigma_gp):
    print(f"{p.name:<12s}  {p.fid:>8.4g}  {s:>10.4g}  {s/abs(p.fid):>12.4f}")

## 4. Read the written output files

In [ ]:
table_path = result["table_path"]
print(f"Forecast table: {table_path}")
print(open(table_path).read())

In [ ]:
scorecard_path = result["scorecard_path"]
print(f"Scorecard: {scorecard_path}")
print(open(scorecard_path).read())

## 5. Inspect the FisherResult object

`result["fisher"]` is a `FisherResult` dataclass (see `fisher.py`) with
attributes: `F` (matrix), `cov` (F⁻¹), `sigma` (same as `sigma_gp`),
`corr` (correlation matrix), `steps` (converged stencil steps per param).

In [ ]:
import matplotlib
matplotlib.use("Agg")   # headless; remove if running in a GUI
import matplotlib.pyplot as plt

fr = result["fisher"]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(np.abs(fr.corr), vmin=0, vmax=1, cmap="viridis")
names = [p.name for p in selected]
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticklabels(names)
plt.colorbar(im, ax=ax, label="|correlation|")
ax.set_title(f"GP-only Fisher correlation (z={cfg_b.redshift})")
fig.tight_layout()
plt.savefig("/tmp/nb01_corr.png", dpi=100)
plt.show()
print("Saved to /tmp/nb01_corr.png")

## 6. Run via the CLI

Everything above can also be run from the shell:

```bash
python scripts/run_pipeline.py \
    --config configs/single_z/example.yaml \
    --mode   gp_only \
    --output-dir /tmp/gp_only_test
```

The CLI writes the same `forecast_table.txt` and `scorecard.md`.

To run all 13 z-bins at once:

```bash
python scripts/run_batch.py \
    --config configs/single_z/example.yaml \
    --mode   gp_only
```

Then inspect the across-z view:

```bash
python scripts/aggregate_z.py --base results/single_z_example
```